# Aula 7 — Introdução ao NLP: classificação de textos

**Notebook do estudante — Projeto Cuidadores**

Vamos transformar linguagem humana em números e classificar mensagens nas categorias **sono**, **alimentação** e **medicação**.

> Os dados são sintéticos, sem dados pessoais, e servem exclusivamente para aprendizagem.


## Objetivos

Ao final, você deverá compreender token, vocabulário, Bag of Words, `CountVectorizer`, separação treino/teste, classificação com Naive Bayes e análise de erros.


## 1. Preparando o ambiente


In [ ]:
import io
import os
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42
print("Ambiente preparado!")


In [ ]:
def carregar_csv(nome_arquivo):
    if os.path.exists(nome_arquivo):
        print(f"Arquivo encontrado: {nome_arquivo}")
        return pd.read_csv(nome_arquivo)
    try:
        from google.colab import files
        print(f"Selecione: {nome_arquivo}")
        enviados = files.upload()
        nome = nome_arquivo if nome_arquivo in enviados else next(iter(enviados))
        return pd.read_csv(io.BytesIO(enviados[nome]))
    except ImportError as erro:
        raise FileNotFoundError(f"Coloque '{nome_arquivo}' junto ao notebook.") from erro


## 2. Carregando e conhecendo o dataset


In [ ]:
dados = carregar_csv("mensagens_cuidadores.csv")
print("Dimensões:", dados.shape)
display(dados.head())
display(dados["categoria"].value_counts().to_frame("registros"))


In [ ]:
dados["categoria"].value_counts().plot(kind="bar", color=["#4338CA", "#0F766E", "#F97316"], rot=0, figsize=(7, 4))
plt.title("Distribuição das categorias")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()


## 3. Regra programada × Machine Learning

Uma regra como `if "almoço" in texto` pode funcionar em casos específicos, mas foi escrita diretamente pelo programador. No ML, fornecemos exemplos rotulados para que o algoritmo aprenda padrões.


In [ ]:
texto_exemplo = "Paciente recusou o almoço".lower()
categoria_regra = "alimentacao" if "almoço" in texto_exemplo else "outra"
print("Resultado da regra:", categoria_regra)


## 4. Token, vocabulário e Bag of Words


In [ ]:
frases_demo = ["sono tranquilo", "remédio após almoço", "sono e almoço"]
vetor_demo = CountVectorizer(lowercase=True)
matriz_demo = vetor_demo.fit_transform(frases_demo)
print("Vocabulário:", vetor_demo.get_feature_names_out().tolist())
display(pd.DataFrame(matriz_demo.toarray(), columns=vetor_demo.get_feature_names_out(), index=frases_demo))


O Bag of Words conta ocorrências. Ele não compreende ironia, contexto ou ordem como uma pessoa. Ainda assim, é uma excelente porta de entrada para NLP.


## 5. Feature, target e divisão treino/teste

Primeiro dividimos os textos; depois aprendemos o vocabulário somente com o treino.


In [ ]:
X_texto = dados["texto"]
y = dados["categoria"]

X_treino_texto, X_teste_texto, y_treino, y_teste = train_test_split(
    X_texto, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Treino:", len(X_treino_texto), "| Teste:", len(X_teste_texto))


## 6. Vetorizando corretamente

- treino: `fit_transform()`;
- teste e novas frases: apenas `transform()`.


In [ ]:
vectorizador = CountVectorizer(lowercase=True)
X_treino = vectorizador.fit_transform(X_treino_texto)
X_teste = vectorizador.transform(X_teste_texto)

print("Formato treino:", X_treino.shape)
print("Formato teste:", X_teste.shape)
print("Palavras no vocabulário:", len(vectorizador.get_feature_names_out()))
print("Primeiras palavras:", vectorizador.get_feature_names_out()[:20])


## 7. Treinando o classificador


In [ ]:
modelo = MultinomialNB()
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)
acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia: {acuracia:.2%}")


## 8. Avaliando com as métricas da Aula 6


In [ ]:
print(classification_report(y_teste, previsoes, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_teste, previsoes, cmap="Purples", xticks_rotation=20)
plt.title("Matriz de confusão — NLP")
plt.tight_layout()
plt.show()


In [ ]:
analise_teste = pd.DataFrame({
    "texto": X_teste_texto.values,
    "categoria_real": y_teste.values,
    "categoria_prevista": previsoes,
})
analise_teste["acertou"] = analise_teste["categoria_real"] == analise_teste["categoria_prevista"]
display(analise_teste)
print("Erros encontrados:", (~analise_teste["acertou"]).sum())


## 9. Testando frases novas com o mesmo vectorizer


In [ ]:
novas_frases = [
    "não conseguiu dormir durante a madrugada",
    "aceitou o jantar e bebeu água",
    "o remédio da noite ficou atrasado",
]
novos_vetores = vectorizador.transform(novas_frases)
novas_previsoes = modelo.predict(novos_vetores)
display(pd.DataFrame({"texto": novas_frases, "previsao": novas_previsoes}))


## 10. Cinco testes externos


In [ ]:
externos = carregar_csv("frases_externas_aula07.csv")
X_externos = vectorizador.transform(externos["texto"])
externos["categoria_prevista"] = modelo.predict(X_externos)
externos["acertou"] = externos["categoria_real"] == externos["categoria_prevista"]
display(externos)
acuracia_externa = externos["acertou"].mean()
print(f"Acurácia externa: {acuracia_externa:.2%}")


### Análise dos erros

Escolha até três frases interessantes e explique:

- havia palavra desconhecida?
- a frase era ambígua?
- faltaram exemplos semelhantes?
- havia mais de um assunto?

**Resposta do grupo:** escreva aqui.


## 11. Desafio — sua própria frase


In [ ]:
# Altere a frase e execute novamente.
FRASE_DO_GRUPO = "dormiu pouco e recusou o café da manhã"
vetor_grupo = vectorizador.transform([FRASE_DO_GRUPO])
previsao_grupo = modelo.predict(vetor_grupo)[0]
print("Frase:", FRASE_DO_GRUPO)
print("Previsão:", previsao_grupo)


Uma frase pode tratar de mais de um assunto. O modelo atual precisa escolher apenas uma categoria; essa limitação deve aparecer na conclusão.


## 12. Exportando os resultados


In [ ]:
analise_teste.to_csv("previsoes_teste_aula07.csv", index=False)
externos.to_csv("previsoes_externas_aula07.csv", index=False)
print("Arquivos de resultados gerados!")


## Conclusão

Responda:

1. Como o texto virou números?
2. O que o vectorizer aprendeu no `fit`?
3. Por que o teste usa apenas `transform`?
4. Quais foram os principais erros?
5. O dataset tem variedade suficiente?
6. Como melhorar o experimento?

**Conclusão do grupo:** escreva aqui.


## Checklist de entrega

- [x] dataset e categorias;
- [x] treino/teste;
- [x] CountVectorizer e vetorização;
- [x] modelo, previsões e métricas;
- [x] cinco testes externos;
- [ ] análise dos erros;
- [ ] conclusão do grupo.
